In [4]:
from pathlib import Path
from urllib.parse import urljoin
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
import time
import random
import json
import threading

# =========================
# MODE SELECTOR
# =========================
MODE = "ET"  # "EAT" or "ET"

if MODE not in {"EAT", "ET"}:
    raise ValueError("MODE must be 'EAT' or 'ET'")

# =========================
# TUNABLES
# =========================
START_YEAR = 2022
END_YEAR = 2026

MAX_THREADS = 20           # <-- define parallelism here
MAX_RETRIES = 4
TIMEOUT = 20

SLEEP_MIN = 0.05
SLEEP_MAX = 0.20

CHECKPOINT_EVERY = 50       # checkpoint every N completed case tasks
ERROR_RATE_THRESHOLD = 0.20 # adaptive throttling trigger
THROTTLE_MULTIPLIER = 1.50  # increase sleep if error rate too high
MAX_SLEEP_CAP = 1.50

# =========================
# URL / PATH CONFIG
# =========================
BASE = "https://www.bailii.org"

PATH_MAP = {
    "EAT": "/uk/cases/UKEAT/",
    "ET": "/uk/cases/UKET/",
}

ROOT_MAP = {
    "EAT": Path("/media/hello/Vault/Tribunals/EAT_Bailii"),
    "ET": Path("/media/hello/Vault/Tribunals/ET_Bailii"),
}

BASE_PATH = PATH_MAP[MODE]
ROOT_DIR = ROOT_MAP[MODE]
ROOT_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = ROOT_DIR / f"{MODE.lower()}_bailii_manifest.json"
TMP_MANIFEST_PATH = ROOT_DIR / f"{MODE.lower()}_bailii_manifest.tmp.json"
FAILED_URLS_PATH = ROOT_DIR / f"{MODE.lower()}_bailii_failed_urls.json"

# =========================
# GLOBALS (THREAD-SAFE)
# =========================
thread_local = threading.local()
adaptive_sleep_lock = threading.Lock()
adaptive_sleep_min = SLEEP_MIN
adaptive_sleep_max = SLEEP_MAX

manifest_lock = threading.Lock()
failures_lock = threading.Lock()

# =========================
# HELPERS
# =========================
def get_session():
    """
    One requests.Session per thread.
    """
    if not hasattr(thread_local, "session"):
        s = requests.Session()
        s.headers.update({
            "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                          "(KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36"
        })
        thread_local.session = s
    return thread_local.session


def atomic_write_json(obj, out_path: Path, tmp_path: Path):
    tmp_path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")
    tmp_path.replace(out_path)


def normalize_url(url: str) -> str:
    return url.strip()


def extract_case_id(url: str) -> str:
    return normalize_url(url).split("/")[-1].replace(".html", "")


def is_valid_case_url(url: str, year: int) -> bool:
    url = normalize_url(url)

    if not url.startswith(BASE):
        return False

    if f"{BASE_PATH}{year}/" not in url:
        return False

    if not url.endswith(".html"):
        return False

    if "?" in url or "#" in url:
        return False

    return True


def fetch(url: str):
    """
    Returns:
        - response.text on 200
        - "NOT_FOUND" on 404
        - None on repeated failure
    """
    global adaptive_sleep_min, adaptive_sleep_max

    session = get_session()
    last_exception = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            with adaptive_sleep_lock:
                sleep_min = adaptive_sleep_min
                sleep_max = adaptive_sleep_max

            time.sleep(random.uniform(sleep_min, sleep_max))

            response = session.get(url, timeout=TIMEOUT)

            if response.status_code == 404:
                return "NOT_FOUND"

            if response.status_code == 200:
                return response.text

            # Retry on transient/server-side issues
            if response.status_code in {429, 500, 502, 503, 504}:
                backoff = 0.6 * attempt
                time.sleep(backoff)
                continue

            return None

        except Exception as e:
            last_exception = e
            time.sleep(0.6 * attempt)

    return None


def extract_case_links(index_html: str, year: int):
    soup = BeautifulSoup(index_html, "html.parser")
    links = set()

    for a in soup.find_all("a"):
        href = a.get("href")
        if not href:
            continue

        full = urljoin(BASE, href)
        full = normalize_url(full)

        if is_valid_case_url(full, year):
            links.add(full)

    return sorted(links)


def clean_html_to_text(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()

    lines = [line.strip() for line in soup.get_text(separator="\n").splitlines()]
    lines = [line for line in lines if line]

    return "\n".join(lines)


def load_json_if_exists(path: Path, default):
    if path.exists():
        try:
            return json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            return default
    return default


def checkpoint(manifest: dict, failed_urls: list):
    with manifest_lock:
        atomic_write_json(manifest, MANIFEST_PATH, TMP_MANIFEST_PATH)

    with failures_lock:
        FAILED_URLS_PATH.write_text(
            json.dumps(failed_urls, indent=2, ensure_ascii=False),
            encoding="utf-8"
        )


def maybe_throttle(results_counter: dict):
    """
    Adaptive throttling:
    If current fail/error share is too high, slow requests a bit.
    """
    global adaptive_sleep_min, adaptive_sleep_max

    total = sum(results_counter.values())
    if total == 0:
        return

    bad = results_counter.get("fail", 0) + results_counter.get("error", 0)
    error_rate = bad / total

    if error_rate >= ERROR_RATE_THRESHOLD:
        with adaptive_sleep_lock:
            adaptive_sleep_min = min(adaptive_sleep_min * THROTTLE_MULTIPLIER, MAX_SLEEP_CAP)
            adaptive_sleep_max = min(adaptive_sleep_max * THROTTLE_MULTIPLIER, MAX_SLEEP_CAP)
    else:
        with adaptive_sleep_lock:
            adaptive_sleep_min = max(SLEEP_MIN, adaptive_sleep_min / 1.10)
            adaptive_sleep_max = max(SLEEP_MAX, adaptive_sleep_max / 1.10)


def process_case(url: str, year_dir: Path):
    """
    Return tuple:
    ("ok"|"skip"|"fail"|"error", payload)
    """
    case_id = extract_case_id(url)

    raw_path = year_dir / f"{case_id}.html"
    text_path = year_dir / f"{case_id}.txt"

    # restart-safe
    if raw_path.exists() and text_path.exists():
        return ("skip", {
            "case_id": case_id,
            "url": url,
            "raw_path": str(raw_path),
            "text_path": str(text_path),
        })

    html = fetch(url)

    if html == "NOT_FOUND":
        return ("fail", {"case_id": case_id, "url": url, "reason": "404"})

    if not html:
        return ("fail", {"case_id": case_id, "url": url, "reason": "fetch_failed"})

    try:
        raw_path.write_text(html, encoding="utf-8")

        clean_text = clean_html_to_text(html)
        text_path.write_text(clean_text, encoding="utf-8")

        title = ""
        try:
            soup = BeautifulSoup(html, "html.parser")
            title = (soup.title.string or "").strip() if soup.title else ""
        except Exception:
            title = ""

        return ("ok", {
            "case_id": case_id,
            "url": url,
            "title": title,
            "raw_path": str(raw_path),
            "text_path": str(text_path),
            "year": year_dir.name,
            "source": "BAILII",
            "mode": MODE,
        })

    except Exception as e:
        return ("error", {
            "case_id": case_id,
            "url": url,
            "reason": repr(e),
        })


def retry_failed_cases(year: int, year_dir: Path, failed_urls_for_year: list, manifest: dict):
    """
    End-of-year retry queue for failed URLs.
    Only retries URLs that do not already exist in manifest with files present.
    """
    if not failed_urls_for_year:
        return {"ok": 0, "skip": 0, "fail": 0, "error": 0}

    results = {"ok": 0, "skip": 0, "fail": 0, "error": 0}

    retry_candidates = []
    for url in failed_urls_for_year:
        case_id = extract_case_id(url)
        rec = manifest.get(case_id)
        if rec:
            raw_ok = Path(rec.get("raw_path", "")).exists()
            text_ok = Path(rec.get("text_path", "")).exists()
            if raw_ok and text_ok:
                continue
        retry_candidates.append(url)

    if not retry_candidates:
        return results

    with ThreadPoolExecutor(max_workers=MAX_THREADS) as executor:
        futures = [executor.submit(process_case, url, year_dir) for url in retry_candidates]

        for future in tqdm(
            as_completed(futures),
            total=len(futures),
            desc=f"{MODE} {year} retry",
            unit="case"
        ):
            status, payload = future.result()
            results[status] += 1

            if status == "ok":
                with manifest_lock:
                    manifest[payload["case_id"]] = payload

    return results


# =========================
# MAIN
# =========================
def main():
    print(f"[{MODE}] Root directory: {ROOT_DIR}")
    print(f"[{MODE}] Manifest path : {MANIFEST_PATH}")
    print(f"[{MODE}] Failed path   : {FAILED_URLS_PATH}")
    print(f"[{MODE}] Years         : {START_YEAR} -> {END_YEAR}")
    print(f"[{MODE}] Max threads   : {MAX_THREADS}")

    manifest = load_json_if_exists(MANIFEST_PATH, default={})
    failed_urls = load_json_if_exists(FAILED_URLS_PATH, default=[])

    print(f"[{MODE}] Loaded manifest records: {len(manifest)}")
    print(f"[{MODE}] Loaded failed URLs     : {len(failed_urls)}")

    all_failed_urls = list(failed_urls)

    for year in range(START_YEAR, END_YEAR + 1):
        print(f"\n=== {MODE} {year} ===")

        year_dir = ROOT_DIR / str(year)
        year_dir.mkdir(parents=True, exist_ok=True)

        index_url = f"{BASE}{BASE_PATH}{year}/"
        index_html = fetch(index_url)

        if index_html == "NOT_FOUND":
            print(f"[{MODE}] {year}: year index does not exist, skipping")
            continue

        if not index_html:
            print(f"[{MODE}] {year}: failed to fetch year index")
            all_failed_urls.append(index_url)
            checkpoint(manifest, sorted(set(all_failed_urls)))
            continue

        case_urls = extract_case_links(index_html, year)
        print(f"[{MODE}] {year}: found {len(case_urls)} candidate case URLs")

        if not case_urls:
            print(f"[{MODE}] {year}: no case URLs found")
            continue

        results = {"ok": 0, "skip": 0, "fail": 0, "error": 0}
        failed_urls_for_year = []
        completed_since_checkpoint = 0

        with ThreadPoolExecutor(max_workers=MAX_THREADS) as executor:
            futures = [executor.submit(process_case, url, year_dir) for url in case_urls]

            for future in tqdm(
                as_completed(futures),
                total=len(futures),
                desc=f"{MODE} {year}",
                unit="case"
            ):
                status, payload = future.result()
                results[status] += 1
                completed_since_checkpoint += 1

                if status == "ok":
                    with manifest_lock:
                        manifest[payload["case_id"]] = payload

                elif status in {"fail", "error"}:
                    failed_url = payload["url"]
                    failed_urls_for_year.append(failed_url)
                    all_failed_urls.append(failed_url)

                maybe_throttle(results)

                if completed_since_checkpoint >= CHECKPOINT_EVERY:
                    checkpoint(manifest, sorted(set(all_failed_urls)))
                    completed_since_checkpoint = 0

        checkpoint(manifest, sorted(set(all_failed_urls)))
        print(f"[{MODE}] {year}: first pass results → {results}")

        # Retry queue for this year
        retry_results = retry_failed_cases(year, year_dir, failed_urls_for_year, manifest)
        checkpoint(manifest, sorted(set(all_failed_urls)))
        print(f"[{MODE}] {year}: retry results      → {retry_results}")

    # Final cleanup of failure list:
    # remove any failed URL whose files now exist in manifest
    final_failed = []
    for url in sorted(set(all_failed_urls)):
        case_id = extract_case_id(url)
        rec = manifest.get(case_id)
        if rec:
            raw_ok = Path(rec.get("raw_path", "")).exists()
            text_ok = Path(rec.get("text_path", "")).exists()
            if raw_ok and text_ok:
                continue
        final_failed.append(url)

    checkpoint(manifest, final_failed)

    print(f"\n[{MODE}] ALL DONE")
    print(f"[{MODE}] Final manifest records: {len(manifest)}")
    print(f"[{MODE}] Final failed URLs     : {len(final_failed)}")


if __name__ == "__main__":
    main()

[ET] Root directory: /media/hello/Vault/Tribunals/ET_Bailii
[ET] Manifest path : /media/hello/Vault/Tribunals/ET_Bailii/et_bailii_manifest.json
[ET] Failed path   : /media/hello/Vault/Tribunals/ET_Bailii/et_bailii_failed_urls.json
[ET] Years         : 2022 -> 2026
[ET] Max threads   : 20
[ET] Loaded manifest records: 42615
[ET] Loaded failed URLs     : 0

=== ET 2022 ===
[ET] 2022: found 6225 candidate case URLs


ET 2022:   0%|          | 0/6225 [00:00<?, ?case/s]

[ET] 2022: first pass results → {'ok': 6225, 'skip': 0, 'fail': 0, 'error': 0}
[ET] 2022: retry results      → {'ok': 0, 'skip': 0, 'fail': 0, 'error': 0}

=== ET 2023 ===
[ET] 2023: found 4177 candidate case URLs


ET 2023:   0%|          | 0/4177 [00:00<?, ?case/s]

[ET] 2023: first pass results → {'ok': 4177, 'skip': 0, 'fail': 0, 'error': 0}
[ET] 2023: retry results      → {'ok': 0, 'skip': 0, 'fail': 0, 'error': 0}

=== ET 2024 ===
[ET] 2024: found 5530 candidate case URLs


...

[ET] 2024: first pass results → {'ok': 5530, 'skip': 0, 'fail': 0, 'error': 0}
[ET] 2024: retry results      → {'ok': 0, 'skip': 0, 'fail': 0, 'error': 0}

=== ET 2025 ===
[ET] 2025: found 6882 candidate case URLs


ET 2025:   0%|          | 0/6882 [00:00<?, ?case/s]

[ET] 2025: first pass results → {'ok': 6882, 'skip': 0, 'fail': 0, 'error': 0}
[ET] 2025: retry results      → {'ok': 0, 'skip': 0, 'fail': 0, 'error': 0}

=== ET 2026 ===
[ET] 2026: found 352 candidate case URLs


ET 2026:   0%|          | 0/352 [00:00<?, ?case/s]

[ET] 2026: first pass results → {'ok': 352, 'skip': 0, 'fail': 0, 'error': 0}
[ET] 2026: retry results      → {'ok': 0, 'skip': 0, 'fail': 0, 'error': 0}

[ET] ALL DONE
[ET] Final manifest records: 65038
[ET] Final failed URLs     : 0
